# Don't Take the Number at Face Value
### Two structure predictors, one impossible protein

**Today's goal:** learn to read the confidence scores that protein structure predictors (AlphaFold2, ESMFold) report — and see for yourself why a confident-looking number is not the same as a true answer.

| Part | Topic | Time |
|---|---|---|
| 0 | Setup — start the GPU and load the model | ~5 min |
| 1 | Build the sequences: a real protein and an impossible one | ~10 min |
| 2 | Model 1 — AlphaFold2 results | ~5 min |
| 3 | One model, two confidence numbers, two answers | ~10 min |
| 4 | Model 2 — run ESMFold yourself | ~15 min |
| 5 | Do the two models agree? | ~10 min |
| 6 | What this does and does not show — discussion | ~15 min |
| ⭐ | Optional extras: per-residue confidence, alanine experiment, saving results | — |

## 📖 How to use this notebook

- **Open it in Google Colab** and choose `File → Save a copy in Drive` first, so your work is kept.
- **Run a cell** by clicking on it and pressing **Shift + Enter** (or the ▶ button on its left). **Run the cells in order, top to bottom** — later cells use results from earlier ones.
- **✏️ TODO** cells contain a blank written as `___`. Replace the `___` (underscores included) with your answer, then run the cell. Each blank is one small thing — usually a single word or number.
- **💡 Hint** lines in the TODO cell tell you what to type.
- **✅ Stuck?** Right below every TODO there is a hidden answer. Click it to reveal. Using it is completely fine — the goal is to understand what the code does, not to type it from memory.
- **⭐ Optional** sections are extras for if you finish early or want to explore after the workshop. You can skip them; nothing later in the notebook depends on them.
- **Lines starting with `#` are comments** written for you to read. Python ignores them.
- **If something goes wrong:** `Runtime → Restart session`, then run the cells from the top again. Or raise your hand — that's what the TAs are for.

No coding background is needed.

⚠️ **This notebook needs a GPU.** Before running anything: `Runtime → Change runtime type → T4 GPU → Save`. If no GPU is available to you, don't worry — the notebook falls back to saved results so you can still follow every step.

---
## The claim we are testing

Not "AI models are bad." Not "AlphaFold hallucinates." Something narrower and much easier to defend:

> A model can hand you a confident-looking number that does not answer the question you are asking. Having "AI" attached does not make the output ground truth, and the way you find out is by checking.

We check in two ways. First, we give a model a protein that **cannot physically exist** and see what confidence it reports. Then we give the same sequences to a **second, completely different model** and see whether the two agree.

### Background: two different confidence numbers
Structure predictors take an amino-acid sequence and return a 3D structure **plus** confidence scores:

- **pLDDT** (0–100) — *per residue*: how accurate is the predicted position of this residue likely to be? Above **70** is conventionally "confident". The headline number is usually the average over all residues.
- **pTM** (0–1) — *whole protein*: is the **overall fold** likely to be right? Below **0.5** conventionally means "probably wrong".

Hold onto the fact that there are two different numbers. It matters later.

---
## Part 0 — Setup: load ESMFold ⏱ ~5 min

👉 **Run this cell first and read Parts 1–3 while it works.** It installs the libraries and downloads the ESMFold model (a few GB), which takes about 2–3 minutes.

You'll see some warnings (e.g. about `HF_TOKEN` or "MISSING" parameters) — those are harmless. When it prints `✅ ESMFold loaded`, you're ready for Part 4.

In [ ]:
!pip install -q transformers accelerate py3Dmol

import time
import numpy as np
import matplotlib.pyplot as plt

MODEL_OK = False     # becomes True if ESMFold loads successfully
try:
    import torch
    from transformers import AutoTokenizer, EsmForProteinFolding

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)
    if device == "cpu":
        raise RuntimeError("No GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")

    # Load in half precision (fp16) to keep memory use low enough for a free Colab session
    model = EsmForProteinFolding.from_pretrained(
        "facebook/esmfold_v1",
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )
    # ...then switch everything EXCEPT the language model back to full precision (fp32).
    # (See the optional "Under the hood" note at the end for why.)
    for name, module in model.named_children():
        if name != "esm":
            module.float()
    for _, param in model.named_parameters(recurse=False):
        param.data = param.data.float()

    model = model.to(device).eval()
    model.trunk.set_chunk_size(64)
    MODEL_OK = True
    print(f"✅ ESMFold loaded in {time.time()-t0:.0f}s")
except Exception as e:
    print(f"⚠️ Could not load ESMFold: {type(e).__name__}: {e}")
    print("No problem — Part 4 will use saved results from an earlier run so you can keep going.")

---
## Part 1 — The Sequences ⏱ ~10 min

**GB1** — the B1 domain of protein G. 56 residues: a four-stranded β-sheet packed against one α-helix, one of the best-characterised small domains in biochemistry. Its **hydrophobic core** is textbook: Y3, L5, L7, F30, A34, V39, W43, Y45, F52 and V54 all point inward and pack together. That packing is essentially the whole reason the domain is stable.

We mutate those core positions to **aspartate (D)**, two at a time. Aspartate carries a negative charge, and burying a charge inside a protein, away from water, costs an enormous amount of energy. Burying ten of them in a 56-residue core is not a marginal call — no structural biologist would defend it. By `core_10D` we have a sequence that **cannot fold**.

Plus two **controls** with *identical amino-acid composition* to wild-type, but scrambled order: **shuffled** and **reversed**.

The whole experiment rests on one fact: every core-mutant stays **82% or more identical to wild-type**, so on the surface they all look like GB1 while the inside becomes impossible.

👉 Run the cell. Each line shows a variant's name, its sequence and its % identity to wild-type. Look for the `D`s appearing.

In [ ]:
WT = "MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE"
CORE = [3, 5, 7, 30, 34, 39, 43, 45, 52, 54]   # buried core positions (counting from 1)
assert len(WT) == 56

def mutate_core(n):
    # Return the GB1 sequence with the first n core positions changed to aspartate (D)
    s = list(WT)
    for p in CORE[:n]:
        s[p-1] = "D"          # p-1 because Python counts from 0
    return "".join(s)

def identity(seq):
    # % of positions identical to wild-type
    return 100 * sum(a == b for a, b in zip(WT, seq)) / len(WT)

rng = np.random.default_rng(0)          # seed 0 = the same shuffled sequence used for the AlphaFold2 run
JOBS = {f"core_{n:02d}D": mutate_core(n) for n in [0, 2, 4, 6, 8, 10]}
JOBS["shuffled"] = "".join(rng.permutation(list(WT)))
JOBS["reversed"] = WT[::-1]

for k, s in JOBS.items():
    print(f"{k:<12} {s}  {identity(s):5.1f}% id")

### ✏️ Make your own variant
Pick how many core positions to mutate — any whole number from 0 to 10. You'll fold this sequence yourself in Part 4.

In [ ]:
# ✏️ TODO: choose how many core residues to change to aspartate (a whole number from 0 to 10).
# 💡 Hint: try 5 — halfway between the real protein and the impossible one
my_n = ___

my_variant = mutate_core(my_n)
print(f"my_variant ({my_n} core D's): {my_variant}")
print(f"identity to wild-type: {identity(my_variant):.1f}%")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
my_n = 5
```

</details>

---
## Part 2 — Model 1: AlphaFold2 (already run) ⏱ ~5 min

AlphaFold2 needs a slow database search for each sequence, so these numbers were computed beforehand with **ColabFold** and pasted in — this keeps the workshop from depending on a live search. AlphaFold2 had every advantage here: a full **multiple sequence alignment (MSA)** of evolutionary relatives, the strongest configuration anyone would ask for.

👉 Run the cell to see the results table.

In [ ]:
AF2 = {
    "core_00D": {"plddt": 94.8, "ptm": 0.767},
    "core_02D": {"plddt": 94.0, "ptm": 0.749},
    "core_04D": {"plddt": 86.5, "ptm": 0.619},
    "core_06D": {"plddt": 88.0, "ptm": 0.640},
    "core_08D": {"plddt": 79.9, "ptm": 0.521},
    "core_10D": {"plddt": 76.9, "ptm": 0.489},
    "shuffled": {"plddt": 45.5, "ptm": 0.207},
    "reversed": {"plddt": 39.7, "ptm": 0.250},
}
print(f"{'job':<12} {'pLDDT':>7} {'pTM':>6}")
for k, v in AF2.items():
    print(f"{k:<12} {v['plddt']:>7.1f} {v['ptm']:>6.3f}")

In Python, the results are stored as a dictionary of dictionaries: `AF2["core_10D"]["plddt"]` means *"in the AF2 results, find core_10D, then its pLDDT"*.

In [ ]:
# ✏️ TODO: look up one variant. Replace the blank with a variant name, in quotes.
# 💡 Hint: try "core_10D" — the protein that cannot fold
k = ___

print(f"{k}:  pLDDT = {AF2[k]['plddt']}   pTM = {AF2[k]['ptm']}")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
k = "core_10D"
```

</details>

---
## Part 3 — One Model, Two Numbers, Two Answers ⏱ ~10 min

Before adding a second model, let's see what a single model already told us. We apply the two conventional thresholds to every variant:
- pLDDT above **70** → "confident"
- pTM above **0.5** → "fold looks right"

In [ ]:
# ✏️ TODO: fill in the two conventional thresholds.
# 💡 Hint: pLDDT threshold is 70, pTM threshold is 0.5
PLDDT_THRESHOLD = ___
PTM_THRESHOLD = ___

for k, v in AF2.items():
    plddt_ok = v["plddt"] > PLDDT_THRESHOLD
    ptm_ok = v["ptm"] > PTM_THRESHOLD
    verdict_1 = "confident" if plddt_ok else "not confident"
    verdict_2 = "fold looks right" if ptm_ok else "fold probably wrong"
    flag = "   <-- the two metrics DISAGREE" if plddt_ok != ptm_ok else ""
    print(f"{k:<10} pLDDT {v['plddt']:5.1f} -> {verdict_1:<14} | pTM {v['ptm']:.3f} -> {verdict_2}{flag}")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
PLDDT_THRESHOLD = 70
PTM_THRESHOLD = 0.5
```

</details>

Now the same thing as a picture: how both numbers change as we bury more and more aspartates.

In [ ]:
import matplotlib.pyplot as plt

ns   = [0,2,4,6,8,10]
keys = [f"core_{n:02d}D" for n in ns]
pl   = [AF2[k]["plddt"] for k in keys]
pt   = [AF2[k]["ptm"]*100 for k in keys]      # scaled to share the axis

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ns, pl, "o-", lw=2.5, ms=9, color="#1f4e8c", label="pLDDT  (threshold 70)")
ax.plot(ns, pt, "s--", lw=2.5, ms=8, color="#c0392b", label="pTM x100  (threshold 50)")
ax.axhline(70, color="#1f4e8c", ls=":", lw=1.5)
ax.axhline(50, color="#c0392b", ls=":", lw=1.5)
ax.axhline(AF2["shuffled"]["plddt"], color="grey", ls="-", lw=1.5)
ax.text(0.1, AF2["shuffled"]["plddt"]+1.5,
        "shuffled = 45.5  ('the model has no idea' floor)", fontsize=9, color="grey")
ax.annotate("cannot fold,\nbut pLDDT still says 'confident'",
            xy=(10, 76.9), xytext=(5.6, 93), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#1f4e8c"))
ax.annotate("pTM has dropped\nbelow its threshold",
            xy=(10, 48.9), xytext=(5.6, 22), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#c0392b"))
ax.set_xlabel("buried core residues mutated to aspartate")
ax.set_ylabel("confidence")
ax.set_ylim(0,100); ax.set_xticks(ns)
ax.legend(loc="lower left", fontsize=10)
ax.set_title("AlphaFold2: two confidence metrics, two different verdicts", pad=12)
plt.tight_layout(); plt.show()

### 🔎 What just happened

For `core_10D`, the protein that cannot exist:

- **pLDDT = 76.9** → above 70, reads as *confident*. This is the number quoted in papers, used in filtering pipelines, and shown in database front-ends.
- **pTM = 0.489** → below 0.5, reads as *the overall fold is probably wrong*.

Same model. Same prediction. Same sequence. The two confidence metrics disagree about whether to trust it — and someone who grabbed the headline number would have grabbed the one that says "fine".

This needs no second model and no claim that AlphaFold2 is broken. It just shows that a number labelled "confidence" has a **specific meaning**, and reading it as a general endorsement is the user's error.

🤔 **Before moving on:** the grey line shows the shuffled sequence (45.5) — what AlphaFold2 outputs when it genuinely has no idea. How far above that "no idea" floor is `core_10D`?

---
## Part 4 — Model 2: ESMFold ⏱ ~15 min

Now the second check, and the reason it is worth doing.

ESMFold is a **different architecture** that uses a **different information source**. AlphaFold2 searches a database for evolutionary relatives and leans heavily on that alignment. ESMFold has no alignment at all — it predicts from the **single sequence**, using a *protein language model* trained on millions of natural protein sequences.

So the two models are not two opinions drawn from the same evidence. They are two genuinely independent reads, which is exactly why comparing them is informative rather than redundant.

**We don't know in advance which way this goes, and that is the point.** If ESMFold agrees with AlphaFold2, the result is much stronger than one model could make it. If it disagrees, the disagreement is itself the lesson: two "AI" systems, same input, different answers, and no way to tell which to believe without checking against something outside both.

### ✍️ Predict first
Before running ESMFold, write down your prediction. *Double-click this cell to edit it.*

**My prediction:** For `core_10D`, ESMFold's pLDDT will be … (higher than / similar to / lower than) AlphaFold2's 76.9, because …

The cell below defines a helper function `esmfold(sequence, name)` that folds one sequence and returns its average pLDDT, its pTM, the per-residue pLDDT and the predicted 3D structure. Then it folds all eight sequences — about 10–20 seconds on a GPU.

In [ ]:
def esmfold(seq, name):
    # Fold one sequence with ESMFold; return its confidence scores and 3D structure
    inp = tokenizer([seq], return_tensors="pt", add_special_tokens=False)
    inp = {k: v.to(device) for k, v in inp.items()}
    with torch.no_grad():
        out = model(**inp)

    plddt = out["plddt"][0, :, 1].detach().float().cpu().numpy()
    if np.isnan(plddt).any():
        raise RuntimeError(f"{name}: NaN in pLDDT -- the model is still running "
                           "in half precision somewhere. Re-run the setup cell.")
    if plddt.max() <= 1.0:
        plddt = plddt * 100.0      # put pLDDT on the 0-100 scale

    # pTM is the fragile one; never let it take down the whole run
    try:
        ptm = float(out["ptm"])
    except Exception as e:
        print(f"   (pTM unavailable for {name}: {type(e).__name__})")
        ptm = float("nan")

    print(f"{name:<12} pLDDT={plddt.mean():5.1f}  pTM={ptm:5.3f}")
    return {"plddt": float(plddt.mean()), "ptm": ptm,
            "per_residue": plddt, "pdb": model.output_to_pdb(out)[0]}

ESM = {}
if MODEL_OK:
    for name, seq in JOBS.items():
        try:
            ESM[name] = esmfold(seq, name)
        except Exception as e:
            print(f"{name:<12} FAILED: {type(e).__name__}: {e}")
else:
    print("ESMFold is not loaded — run the next (backup) cell.")

🛟 **Backup cell — safe to run either way.** If ESMFold worked above, this does nothing. If it didn't (no GPU, out of memory…), it fills in the results from an earlier ESMFold run so you can continue. *(Backup results have no 3D structure or per-residue scores, so the 3D viewer below will be skipped.)*

In [ ]:
ESM_BACKUP = {   # mean pLDDT and pTM from an earlier ESMFold run on the same eight sequences
    "core_00D": {"plddt": 89.0, "ptm": 0.761},
    "core_02D": {"plddt": 64.2, "ptm": 0.454},
    "core_04D": {"plddt": 52.4, "ptm": 0.316},
    "core_06D": {"plddt": 50.6, "ptm": 0.278},
    "core_08D": {"plddt": 48.2, "ptm": 0.111},
    "core_10D": {"plddt": 55.0, "ptm": 0.110},
    "shuffled": {"plddt": 49.2, "ptm": 0.195},
    "reversed": {"plddt": 43.5, "ptm": 0.194},
}
missing = [k for k in JOBS if k not in ESM]
if not missing:
    print("ESMFold ran fine — no backup needed.")
else:
    for k in missing:
        ESM[k] = dict(ESM_BACKUP[k])
    print("Filled in backup results for:", ", ".join(missing))

### ✏️ Fold your own variant
Remember `my_variant` from Part 1? Fold it with the same `esmfold` function. *(Needs ESMFold to be loaded.)*

In [ ]:
if MODEL_OK:
    # ✏️ TODO: fold your variant. Replace the blank with the variable holding your sequence.
    # 💡 Hint: my_variant
    ESM["my_variant"] = esmfold(___, "my_variant")

    print(f"\nYour variant has {my_n} buried aspartates.")
    print("Compare its pLDDT and pTM with the core_XXD variants on either side of it.")
else:
    print("ESMFold isn't loaded, so we can't fold new sequences — skip this one.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
ESM["my_variant"] = esmfold(my_variant, "my_variant")
```

</details>

---
## Part 5 — Do the Two Models Agree? ⏱ ~10 min

👉 Look back at your prediction, then run the comparison. Read the table **before** drawing any conclusion.

The `diff` column is ESMFold minus AlphaFold2: negative means ESMFold was *less* confident.

In [ ]:
print(f"{'job':<12} {'AF2 pLDDT':>10} {'ESM pLDDT':>10} {'diff':>7}   {'AF2 pTM':>8} {'ESM pTM':>8}")
print("-"*62)
for k in JOBS:
    a, e = AF2[k], ESM[k]
    print(f"{k:<12} {a['plddt']:>10.1f} {e['plddt']:>10.1f} {e['plddt']-a['plddt']:>7.1f}"
          f"   {a['ptm']:>8.3f} {e['ptm']:>8.3f}")

fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(JOBS)); w = 0.38
names = list(JOBS)
ax.bar(x-w/2, [AF2[k]["plddt"] for k in names], w, label="AlphaFold2 (with MSA)", color="#1f4e8c")
ax.bar(x+w/2, [ESM[k]["plddt"] for k in names], w, label="ESMFold (single sequence)", color="#e67e22")
ax.axhline(70, color="grey", ls="--", lw=1.2)
ax.text(len(names)-0.5, 71.5, "confident", fontsize=8, color="grey", ha="right")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right")
ax.set_ylabel("mean pLDDT"); ax.set_ylim(0,100)
ax.legend(fontsize=9)
ax.set_title("Two models, two information sources, same eight sequences", pad=12)
plt.tight_layout(); plt.show()

### Look at the structures
Seeing is believing. The viewer colours each residue by its pLDDT: **red/orange = low confidence**, **green/blue = high confidence**. You can rotate the structure with your mouse and zoom with the scroll wheel.

In [ ]:
import py3Dmol

def show(pdb, w=360, h=320):
    v = py3Dmol.view(width=w, height=h)
    v.addModel(pdb, "pdb")
    v.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
    v.zoomTo()
    return v

# ✏️ TODO: pick a second variant to view next to wild-type (core_00D).
# 💡 Hint: "core_10D"  — or "shuffled", or "my_variant" if you folded it
for k in ["core_00D", ___]:
    if "pdb" in ESM.get(k, {}):
        print(f"\n=== ESMFold: {k} — mean pLDDT {ESM[k]['plddt']:.1f} ===")
        show(ESM[k]["pdb"]).show()
    else:
        print(f"No 3D structure available for {k} (backup results only).")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
for k in ["core_00D", "core_10D"]:
```

</details>

🗣️ **Discuss with your neighbour:**
- Where did the two models agree, and where did they disagree?
- Which model's confidence looks closer to what the chemistry tells us about `core_10D`?
- If you only had one of these two models, what would you have concluded?

---
## Part 6 — What This Does and Does Not Show ⏱ ~15 min

### ✅ What we are entitled to say

**A sequence that cannot fold did not get a "cannot fold" answer.** By `core_10D` there are ten buried negative charges in a 56-residue core. AlphaFold2 returned pLDDT 76.9 — above every threshold in common use, and thirty points above the 45.5 it gives the shuffled sequence, which is what it outputs when it genuinely has nothing. The model's confidence moved in the right direction and stopped far short of where the chemistry actually is.

**The two confidence metrics disagreed.** pLDDT said confident, pTM said probably wrong. Picking the headline number would have picked the wrong one.

**Whatever the second model did is also data.** If ESMFold tracked AlphaFold2, two independent systems share the limitation and it isn't an artifact of one pipeline. If it diverged, you have two "AI" answers to the same question that cannot both be right — and nothing inside either model tells you which.

### ❌ What we are *not* entitled to say

**Not "AlphaFold2 is broken."** It was trained on the Protein Data Bank (PDB), which contains only proteins that fold. It was never shown an unfoldable sequence and asked to flag one. pLDDT is *predicted accuracy*, not *predicted stability*. On its own terms the model behaved correctly.

**Not "the model is blind to chemistry."** The curve does decline, 94.8 to 76.9. It noticed something. It understated it badly, which is a different and smaller claim.

**Not "this is about LLMs."** AlphaFold2 is not a language model. ESMFold contains one. If the two behave similarly here, that is evidence the issue isn't specific to language models at all.

### 💡 The actual lesson

> A confidence score reports expected performance **on the task the model was trained for**. Read as an answer to any other question, it is not evidence — it just looks like evidence, because it is a number between 0 and 100 with the word "confidence" attached.

The failure is in the *inference*, not the model. And the way you catch it is exactly what we did: **build a case where you already know the answer, and check whether the model's number tracks it.** That works on any model, in any field, without needing to understand its internals.

### 🗣️ Questions for discussion
- pLDDT does not measure foldability. What would? What experiment gives ground truth? *(See Tsuboyama et al. 2023 — hundreds of thousands of measured folding stabilities.)*
- Shuffled scored 45.5 with AlphaFold2. Is that the model reasoning about chemistry, or a database search returning nothing? How would you tell those apart?
- If two models disagree, how do you decide? What breaks the tie?
- Where else — in your own research — do you read a confidence score as an answer to a question the model was not trained on?

### 📝 A terminology note
In the protein field, **"hallucination" already means something specific**: deliberately designing a sequence to maximise a network's confidence (Anishchenko et al. 2021). It is a *design method*, not a failure. What we saw today is better called **miscalibrated confidence** or **out-of-distribution failure**. Using "hallucination" with a protein audience sends people down the wrong path.

---
## ⭐ Optional — Where along the chain does confidence drop?

> **Optional:** for if you finish early. Needs ESMFold to have run (not the backup).

The headline pLDDT is an *average*. Plotting it residue by residue shows *where* the model is unsure. The red bands mark the 10 core positions we mutated.

In [ ]:
# ✏️ TODO: choose a second variant to compare against wild-type.
# 💡 Hint: "core_10D"
k1 = "core_00D"
k2 = ___

if "per_residue" in ESM.get(k1, {}) and "per_residue" in ESM.get(k2, {}):
    positions = range(1, len(WT) + 1)
    plt.figure(figsize=(10, 4))
    plt.plot(positions, ESM[k1]["per_residue"], "o-", ms=3, label=k1)
    plt.plot(positions, ESM[k2]["per_residue"], "o-", ms=3, label=k2)
    for p in CORE:
        plt.axvspan(p - 0.5, p + 0.5, color="red", alpha=0.12)
    plt.axhline(70, color="grey", ls="--", lw=1)
    plt.xlabel("residue position"); plt.ylabel("ESMFold pLDDT"); plt.ylim(0, 100)
    plt.title("Per-residue confidence (red bands = mutated core positions)")
    plt.legend(); plt.tight_layout(); plt.show()
else:
    print("Per-residue scores need a live ESMFold run (not available from the backup).")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
k2 = "core_10D"
```

</details>

---
## ⭐ Optional — A milder experiment: core to alanine

> **Optional:** needs ESMFold to be loaded.

Aspartate is an extreme choice: it removes the packing **and** buries a charge. **Alanine (A)** is small and uncharged — it removes the packing but buries no charge. Does ESMFold's confidence fall less steeply for alanine than for aspartate? What would you predict?

In [ ]:
def mutate_core_to(n, aa):
    # Like mutate_core, but replace the first n core positions with any amino acid letter
    s = list(WT)
    for p in CORE[:n]:
        s[p-1] = aa
    return "".join(s)

# ✏️ TODO: choose the replacement amino acid (one capital letter, in quotes).
# 💡 Hint: "A" for alanine
new_aa = ___

if MODEL_OK:
    for n in [0, 4, 10]:
        name = f"core_{n:02d}{new_aa}"
        ESM[name] = esmfold(mutate_core_to(n, new_aa), name)
    print("\nCompare with the aspartate series: core_04D =",
          f"{ESM['core_04D']['plddt']:.1f}, core_10D = {ESM['core_10D']['plddt']:.1f}")
else:
    print("ESMFold isn't loaded, so we can't fold new sequences.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
new_aa = "A"
```

</details>

---
## ⭐ Optional — Save your results

Saves both models' scores (and the ESMFold structures) to a file you can download and reopen later.

In [ ]:
import pickle
slim = {k: {"plddt": v["plddt"], "ptm": v["ptm"], "pdb": v.get("pdb")} for k, v in ESM.items()}
with open("both_models.pkl", "wb") as f:
    pickle.dump({"af2": AF2, "esm": slim}, f)
print("saved both_models.pkl")

# To download it, remove the # from the next line and run again:
# from google.colab import files; files.download("both_models.pkl")

---
## ⭐ Optional — Under the hood: why the model is loaded that way

ESMFold is big. Loading it at full precision (fp32) and converting afterwards can exceed the RAM of a free Colab session ("Your session crashed after using all available RAM"). Loading directly in **half precision (fp16)** roughly halves the memory peak.

But the folding trunk and the pLDDT/pTM heads don't work well in half precision: the pTM head produces `NaN` (not-a-number) and can crash. So the setup cell keeps only the **language model** in fp16 and switches **everything else** back to fp32. This reproduces ESMFold's documented configuration while avoiding the memory crash.

---
## Limitations of this demo, stated plainly

- **AlphaFold2 without an alignment is not a usable control.** In an earlier run, wild-type GB1 scored 58.5 without an MSA and `core_10D` scored 61.2 — the impossible protein beat the real one. That is noise from a mode that barely works, not a finding. ESMFold is the right way to ask the no-alignment question, which is why it is here.
- **The AlphaFold2 numbers used `--num-models 1 --num-recycle 1`**, which is why the curve wobbles at 4→6 (86.5 then 88.0). For anything presented as a result, re-run with `--num-models 3 --num-recycle 3`.
- **n = 1 protein.** This is a demonstration, not a study. GB1 is one domain; the honest version of this claim needs large-scale stability data like Tsuboyama et al. 2023.

## Sources
- Jumper et al. 2021, *Nature* — AlphaFold2; pLDDT is predicted lDDT-Cα, an accuracy estimate
- Lin et al. 2023, *Science* — ESMFold, single-sequence structure prediction
- Mirdita et al. 2022, *Nature Methods* — ColabFold
- Pak et al. 2023, *PLoS ONE* — AlphaFold2 does not track single-mutation effects on stability
- Tsuboyama et al. 2023, *Nature* — mega-scale measurement of protein folding stability
- Anishchenko et al. 2021, *Nature* — "hallucination" as a deliberate protein design method